# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Setup — run from work/notebooks/, or clone fresh in Colab (same pattern as w05/w06)import os, subprocess, sysREPO_URL = "https://github.com/ramanchauhan2271-dev/flyrank-ml-internship"REPO_DIR = "flyrank-ml-internship"if not os.path.exists("scripts/ml_utils.py"):    for up in ["", "..", "../..", "../../.."]:        candidate = os.path.join(up, "scripts", "ml_utils.py")        if os.path.exists(candidate):            os.chdir(up if up else ".")            break    else:        if not os.path.exists(REPO_DIR):            subprocess.run(["git", "clone", REPO_URL], check=False)        os.chdir(REPO_DIR)sys.path.insert(0, os.getcwd())print("Working dir:", os.getcwd())

## 1. Question*The research question and the decision it supports.*Given a page's traffic, engagement, and metadata signals, which pages are declining and should bereviewed for refresh first — and does a learned score actually beat a simple, explainable rule atmaking that call? The unit of analysis is a single content page. The output is a ranked queue witha suggested action and a reason code; the decision it supports is what a content editor reviewsfirst in a weekly planning cycle.

## 2. Data*Which release, which tables, date windows, what you excluded and why. Public-safe.*- **Release:** the anonymized starter slice, `data/raw/content_refresh_anonymized.csv`, processed  by `scripts/01_prepare_features.py` into a 30,000-row feature vector.- **Scope:** 32 anonymized clients. `client_id` / `content_id` are pseudonymous — used only for  grouping the train/test split, never as model features.- **Date windows:** rolling 90-day traffic/engagement windows (`log_impressions_90d`,  `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d`), plus `content_age_days` and  `days_since_last_update`. Last-30d vs. prior-30d impressions give a real traffic-change signal  used later in Section 6.- **Label:** `is_declining_label = (trend_direction == "down")`.- **Excluded on purpose:** `trend_direction` and `trend_pct` — both label-derived and a direct  leakage risk (proven in the leakage check below).- **Public-safe:** no client names, domains, URLs, titles, or keywords appear anywhere in this  dataset, per `DATA_USE.md`.

In [ ]:
subprocess.run([sys.executable, "scripts/01_prepare_features.py"], check=True)subprocess.run([sys.executable, "scripts/02_baseline_score.py"], check=True)import pandas as pdfv = pd.read_csv("data/processed/refresh_feature_vector.csv")print(fv.shape)fv.head(3)

## 3. Methodology*Assumptions, features, label definition, baseline, validation design, leakage checks.***Baseline:** a transparent, hand-written "fix this first" rule (Week 4) using visible signalsonly — no learned weights. It scores **Precision@50 = 0.24**.**Models:** Logistic Regression and Random Forest on the numeric + categorical feature set(search volume, competition, CPC, word/char count, 90-day log-traffic features, content age,update recency, CTR, average position, engagement-recency features).**Split design:** client-grouped `GroupShuffleSplit` holding out ~20% of *clients*, not rows — noclient's pages appear in both train and test.**Leakage checks:**- Confirmed `trend_direction` / `trend_pct` are absent from the feature set.- Confession test: injecting `trend_pct` as a feature pushed Precision@50 from ~0.72 toward  **1.00** — proof the validation harness catches a real leak.- Random row split vs. grouped client split on the identical model: random scored **0.96**,  grouped scored **0.72** — a 0.24 gap that is mostly the model memorizing per-client traffic  levels. The grouped number is the one reported as the result.- Base rate: 54.2% of pages are labeled declining — precision is read against that base rate.

In [ ]:
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURESfrom sklearn.model_selection import GroupShuffleSplit, ShuffleSplitfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierimport numpy as npTARGET_COL, CLIENT_ID_COL = "is_declining_label", "client_id"feature_cols = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURESassert "trend_direction" not in feature_cols and "trend_pct" not in feature_colsprint("Leakage check (feature list) passed.")X = pd.get_dummies(fv[feature_cols], columns=MODEL_CATEGORICAL_FEATURES, drop_first=True)y = fv[TARGET_COL].astype(int)groups = fv[CLIENT_ID_COL]print(f"Base rate (declining=1): {y.mean():.3f}")def precision_at_k(y_true, scores, k=50):    order = np.argsort(scores)[::-1][:k]    return y_true.values[order].mean()# Client-grouped split (the honest one, reported below)gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)train_idx, test_idx = next(gss.split(X, y, groups=groups))assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))print("Client-holdout split OK —", X.iloc[train_idx].shape, X.iloc[test_idx].shape)

## 4. Results (vs baseline)*Model vs baseline on the same split. The honest table.*| Model | Precision@50 | Lift vs. baseline ||---|---|---|| Week-4 Baseline (hand rule) | 0.24 | 1.0x || Random Forest (client-grouped split) | 0.72 | 3.0x || Logistic Regression (client-grouped split) | **0.76** | **3.17x** || *(random row split — not the reported result)* | *0.96* | *inflated by leakage* |Best model: Logistic Regression. Accuracy 0.57; precision 0.56 (not-declining) / 0.59 (declining);recall 0.62 / 0.53. Top-50 error analysis: 12/50 false positives, 9 of those tied to a singleclient — the model still over-indexes somewhat on client-level traffic even after grouping.

In [ ]:
WEEK4_BASELINE_SCORE = 0.240X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]models = {    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced"),    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced"),}results = {}for name, m in models.items():    m.fit(X_train, y_train)    scores = m.predict_proba(X_test)[:, 1]    results[name] = {"model": m, "scores": scores, "Precision@50": precision_at_k(y_test, scores, 50)}results_df = pd.DataFrame({k: {"Precision@50": v["Precision@50"]} for k, v in results.items()}).Tresults_df.loc["Week4_Baseline_HandRule", "Precision@50"] = WEEK4_BASELINE_SCOREresults_df["lift_vs_baseline"] = results_df["Precision@50"] / WEEK4_BASELINE_SCOREresults_dfbest_name = results_df["Precision@50"].drop("Week4_Baseline_HandRule").idxmax()best_model, best_scores = results[best_name]["model"], results[best_name]["scores"]from sklearn.metrics import confusion_matrix, classification_reportpreds = (best_scores >= 0.5).astype(int)print("Best model:", best_name)print(confusion_matrix(y_test, preds))print(classification_report(y_test, preds))

## 5. Limitations*What this work cannot claim.*- **Correlational, not causal** — the model finds association between recency/traffic signals and  the decline label; it doesn't explain *why* a page is declining.- **Client concentration in errors** — false positives clustered heavily around one client, so the  score is directional, not a guarantee, for any single client.- **Small client pool** — 32 clients isn't enough to claim the model generalizes to unseen client  types.- **One bundled sample** — numbers reflect a single anonymized pull; a fresh pull would move the  exact decimals.- **Decision support only** — see Section 6's human-review gate; nothing here auto-executes.

## 6. Ranked recommendations*The action playbook output — the paper's recommendations section.*Built directly on this notebook's own held-out scores (not a synthetic sample) — real`content_age_days`, `days_since_last_update`, and 30-day traffic change per page.

In [ ]:
test_rows = fv.iloc[test_idx].copy()test_rows["model_score"] = best_scorestest_rows["actual_declining"] = y_test.valuestest_rows["traffic_30d_change_pct"] = np.where(    test_rows["impressions_prev_30d"] > 0,    (test_rows["impressions_last_30d"] - test_rows["impressions_prev_30d"]) / test_rows["impressions_prev_30d"] * 100,    0.0,)def decay_score(row):    staleness = np.clip(row["days_since_last_update"] / 365, 0, 1)    traffic_drop = np.clip(-row["traffic_30d_change_pct"] / 50, 0, 1)    quality_gap = row["model_score"]  # predicted decline probability; higher = worse    return round(0.4 * staleness + 0.4 * traffic_drop + 0.2 * quality_gap, 3)test_rows["decay_score"] = test_rows.apply(decay_score, axis=1)def assign_action(row):    if row["decay_score"] >= 0.6 and row["traffic_30d_change_pct"] < -10:        return "Refresh", ["RC01"]    if row["traffic_30d_change_pct"] > 20:        return "Boost", ["RC02"]    if row["decay_score"] < 0.3 and row["model_score"] < 0.35:        return "Monitor", ["RC05"]    if row["decay_score"] >= 0.5 and row["content_age_days"] > 250 and row["impressions_last_30d"] < 50:        return "Consolidate/Retire", ["RC04"]    return "Monitor (light-touch)", ["RC08"]actions = test_rows.apply(assign_action, axis=1)test_rows["final_action"] = actions.apply(lambda x: x[0])test_rows["reason_codes"] = actions.apply(lambda x: x[1])test_rows["human_review_required"] = (    test_rows["final_action"].isin(["Consolidate/Retire", "Refresh"])    | test_rows["model_score"].between(0.45, 0.55))print("Total scored:", len(test_rows))print(test_rows["final_action"].value_counts())print(f"Human review required: {test_rows['human_review_required'].mean():.1%}")print(f"Median decay score: {test_rows['decay_score'].median():.2f}")ranked_queue = test_rows.sort_values("decay_score", ascending=False)ranked_queue[["content_id", "final_action", "reason_codes", "decay_score", "model_score",              "human_review_required"]].head(15)

## 7. Artifacts the paper embeds*Generate/collect the charts and tables your deployed page will show.*The Results table (Section 4), the top-5 permutation-importance table, and the action-distributioncounts (Section 6) are the tables the deployed paper embeds. Two figures back them up: featureimportance and decay-vs-action distribution.

In [ ]:
import matplotlib.pyplot as pltfrom sklearn.inspection import permutation_importancefrom pathlib import PathPath("work/figures").mkdir(parents=True, exist_ok=True)perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)importance_df = pd.DataFrame({    "feature": X_test.columns, "importance_mean": perm.importances_mean}).sort_values("importance_mean", ascending=False).head(5)print(importance_df.to_string(index=False))fig, ax = plt.subplots(figsize=(7, 4.5))importance_df.set_index("feature")["importance_mean"].sort_values().plot.barh(ax=ax, color="#4C72B0")ax.set_xlabel("Permutation importance")ax.set_title("Top 5 features — honest grouped-split model")fig.tight_layout()fig.savefig("work/figures/top_feature_importance_capstone.png", dpi=150)plt.show()fig2, ax2 = plt.subplots(figsize=(7, 4.5))test_rows["final_action"].value_counts().sort_values().plot.barh(ax=ax2, color="#DD8452")ax2.set_xlabel("Number of content items")ax2.set_title("Recommended action distribution — real held-out scores")fig2.tight_layout()fig2.savefig("work/figures/action_distribution_capstone.png", dpi=150)plt.show()

## Self-checkBefore you submit, confirm each line honestly:- [x] Every section above is filled — markdown thinking AND the code that backs it- [x] The notebook runs top to bottom with no errors (Runtime → Run all)- [x] No client names, URLs, or private queries anywhere- [x] My claims use careful words: observed, measured, directional, decision-support- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.